# GitStars Notebook

Interactive workflow for browsing, filtering, and curating your GitHub starred repositories.

## 1. Setup

Set `GITHUB_TOKEN` in your shell before starting Jupyter, or paste a token into `TOKEN` below for temporary local use.

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.error
import urllib.parse
import urllib.request
import webbrowser
from base64 import b64decode
from pathlib import Path
from IPython.display import Markdown, display

API_BASE = "https://api.github.com"
TOKEN = os.environ.get("GITHUB_TOKEN", "")
USERNAME = ""  # Leave blank to use the account behind the token
QUERY = ""
LANGUAGE = ""
MIN_STARS = 0
SORT_BY = "stars"  # name, stars, updated
TOP_N = 20
DOWNLOAD_DIR = Path("downloads")
CLONE_DIR = Path("clones")


In [ ]:
def build_request(url: str, token: str) -> urllib.request.Request:
    return urllib.request.Request(
        url,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "User-Agent": "gitstars-notebook",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )


def api_get(url: str, token: str):
    request = build_request(url, token)
    try:
        with urllib.request.urlopen(request) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"GitHub API error {exc.code}: {message}") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Network error: {exc.reason}") from exc


def resolve_username(token: str, username: str) -> str:
    if username:
        return username
    payload = api_get(f"{API_BASE}/user", token)
    return payload["login"]


def fetch_starred(username: str, token: str) -> list[dict]:
    repos = []
    page = 1
    while True:
        query = urllib.parse.urlencode({"per_page": 100, "page": page})
        payload = api_get(f"{API_BASE}/users/{username}/starred?{query}", token)
        if not payload:
            break
        repos.extend(payload)
        page += 1
    return repos


def curate_repos(repos: list[dict], query: str = "", language: str = "", min_stars: int = 0, sort_by: str = "stars", top_n: int = 20) -> list[dict]:
    curated = repos
    if query:
        needle = query.lower()
        curated = [
            repo for repo in curated
            if needle in " ".join([
                repo.get("full_name", ""),
                repo.get("description") or "",
                repo.get("language") or "",
                " ".join(repo.get("topics") or []),
            ]).lower()
        ]
    if language:
        curated = [repo for repo in curated if (repo.get("language") or "").lower() == language.lower()]
    if min_stars:
        curated = [repo for repo in curated if (repo.get("stargazers_count") or 0) >= min_stars]

    if sort_by == "name":
        curated = sorted(curated, key=lambda repo: repo.get("full_name", "").lower())
    elif sort_by == "updated":
        curated = sorted(curated, key=lambda repo: repo.get("updated_at") or "", reverse=True)
    else:
        curated = sorted(curated, key=lambda repo: repo.get("stargazers_count") or 0, reverse=True)

    return curated[:top_n] if top_n else curated


def show_repos(repos: list[dict]) -> None:
    for index, repo in enumerate(repos, start=1):
        language = f" [{repo.get('language')}]" if repo.get("language") else ""
        description = f" - {repo.get('description')}" if repo.get("description") else ""
        print(f"{index:>3}. {repo.get('full_name')}{language}{description}")
        print(f"     {repo.get('html_url')}")


def fetch_readme(repo: dict, token: str) -> str:
    payload = api_get(f"{API_BASE}/repos/{repo['full_name']}/readme", token)
    return b64decode(payload["content"]).decode("utf-8", errors="replace")


def preview_readme(repo: dict, token: str) -> None:
    display(Markdown(fetch_readme(repo, token)))


def save_readme(repo: dict, token: str, output_path: str | Path | None = None) -> Path:
    destination = Path(output_path or f"{repo['name']}-README.md")
    destination.write_text(fetch_readme(repo, token), encoding="utf-8")
    return destination


def download_repo(repo: dict, output_dir: str | Path = DOWNLOAD_DIR) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    default_branch = repo.get("default_branch") or "main"
    destination = output_dir / f"{repo['full_name'].replace('/', '-')}-{default_branch}.zip"
    archive_url = f"https://github.com/{repo['full_name']}/archive/refs/heads/{default_branch}.zip"
    with urllib.request.urlopen(archive_url) as response, open(destination, "wb") as handle:
        shutil.copyfileobj(response, handle)
    return destination


def clone_repo(repo: dict, output_dir: str | Path = CLONE_DIR) -> Path:
    if not shutil.which("git"):
        raise RuntimeError("git is not installed or not on PATH.")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / repo["name"]
    if destination.exists():
        raise RuntimeError(f"Clone target already exists: {destination}")
    subprocess.run(["git", "clone", repo["clone_url"], str(destination)], check=True)
    return destination


In [ ]:
if not TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN before running the notebook.")

resolved_username = resolve_username(TOKEN, USERNAME)
all_repos = fetch_starred(resolved_username, TOKEN)
curated_repos = curate_repos(
    all_repos,
    query=QUERY,
    language=LANGUAGE,
    min_stars=MIN_STARS,
    sort_by=SORT_BY,
    top_n=TOP_N,
)

print(f"Fetched {len(all_repos)} starred repos for {resolved_username}")
print(f"Showing {len(curated_repos)} curated repos")
show_repos(curated_repos)

## 2. Inspect one repo

Change `SELECTED_INDEX` to the numbered result you want to inspect.

In [ ]:
SELECTED_INDEX = 1
selected_repo = curated_repos[SELECTED_INDEX - 1]
selected_repo

In [ ]:
preview_readme(selected_repo, TOKEN)

## 3. Actions

Run only the action you want.

In [ ]:
# Open in browser
webbrowser.open(selected_repo["html_url"])

In [ ]:
# Save README locally
save_readme(selected_repo, TOKEN)

In [ ]:
# Download repo as ZIP
download_repo(selected_repo)

In [ ]:
# Clone repo locally
clone_repo(selected_repo)